# Merge & Preprocess Reviews

**Step 1** — Merge `agoda-review-en-vi.csv` + `googlemaps-review-en-vi.csv` into a slim `data-review-en-vi.csv`  
**Step 2** — Normalize + word-segment with `Preprocessor`, split by language → `final-reviews-en.csv` / `final-reviews-vi.csv`

In [ ]:
import sys
from pathlib import Path

import pandas as pd

# Paths
HERE          = Path(".")
AGODA_PATH    = HERE / "agoda"      / "agoda-review-en-vi.csv"
GMAP_PATH     = HERE / "googlemaps" / "googlemaps-review-en-vi.csv"
MERGED_OUT    = HERE / "data-review-en-vi.csv"
EN_OUT        = HERE / "final-reviews-en.csv"
VI_OUT        = HERE / "final-reviews-vi.csv"

# Make preprocessor importable from this directory
sys.path.insert(0, str(HERE))

print("Agoda  :", AGODA_PATH.exists())
print("GMap   :", GMAP_PATH.exists())

## Step 1 — Merge

### 1a. Load & reshape Agoda

In [ ]:
agoda_raw = pd.read_csv(AGODA_PATH, encoding="utf-8-sig", low_memory=False)
print(f"Agoda raw shape: {agoda_raw.shape}")

agoda = pd.DataFrame({
    "review_text"  : agoda_raw["comment"],
    "language"     : agoda_raw["language"],
    # Normalise 1–10 → 1–5
    "rating"       : agoda_raw["score"] / 2,
    "review_year"  : agoda_raw["stay_year"],
    "review_month" : agoda_raw["stay_month"],
    "review_period": agoda_raw["stay_period"],
    "stay_nights"  : agoda_raw["stay_nights"],
    "hotel_id"     : agoda_raw["hotel_id"],
    "hotel_name"   : agoda_raw["hotel_name"],
    "source"       : "agoda",
})

print(f"Agoda reshaped : {agoda.shape}")
agoda.head(2)

### 1b. Load & reshape Google Maps

In [ ]:
gmap_raw = pd.read_csv(GMAP_PATH, encoding="utf-8-sig", low_memory=False)
print(f"GMap raw shape: {gmap_raw.shape}")

gmap = pd.DataFrame({
    "review_text"  : gmap_raw["review_text"],
    "language"     : gmap_raw["review_lang"],
    "rating"       : gmap_raw["rating"].astype(float),
    "review_year"  : gmap_raw["review_year"],
    "review_month" : gmap_raw["review_month"],
    "review_period": gmap_raw["review_period"],
    "stay_nights"  : None,          # not available in Google Maps data
    "hotel_id"     : gmap_raw["hotel_id"],
    "hotel_name"   : gmap_raw["hotel_name"],
    "source"       : "googlemaps",
})

print(f"GMap reshaped  : {gmap.shape}")
gmap.head(2)

### 1c. Concatenate & clean

In [ ]:
df = pd.concat([agoda, gmap], ignore_index=True)
print(f"Combined shape (before clean): {df.shape}")

# Drop rows with null or blank review_text
df = df.dropna(subset=["review_text"])
df = df[df["review_text"].str.strip().astype(bool)].reset_index(drop=True)
df["review_text"] = df["review_text"].str.strip()

print(f"Combined shape (after clean) : {df.shape}")
print()
print("Language breakdown:")
print(df["language"].value_counts())
print()
print("Source breakdown:")
print(df["source"].value_counts())
print()
print("Rating stats:")
print(df["rating"].describe().round(2))

df.head(3)

### 1d. Save `data-review-en-vi.csv`

In [ ]:
df.to_csv(MERGED_OUT, index=False, encoding="utf-8-sig")
print(f"Saved → {MERGED_OUT}")
print(f"Shape : {df.shape}")
print(f"Columns: {df.columns.tolist()}")

## Step 2 — Preprocess

Uses `dataset-prepare/preprocessor.py` → adds `processed_text` column, then splits by language.

In [ ]:
from preprocessor import Preprocessor

pre = Preprocessor()

In [ ]:
df_processed = pre.process(df)

print(f"Shape after preprocessing: {df_processed.shape}")
df_processed[["review_text", "language", "processed_text"]].head(5)

### 2a. Split & save by language

In [ ]:
en_df = df_processed[df_processed["language"] == "en"].reset_index(drop=True)
vi_df = df_processed[df_processed["language"] == "vi"].reset_index(drop=True)

print(f"English reviews : {len(en_df):,}")
print(f"Vietnamese reviews: {len(vi_df):,}")

In [ ]:
en_df.to_csv(EN_OUT, index=False, encoding="utf-8-sig")
vi_df.to_csv(VI_OUT, index=False, encoding="utf-8-sig")

print(f"Saved → {EN_OUT}  ({len(en_df):,} rows)")
print(f"Saved → {VI_OUT}  ({len(vi_df):,} rows)")

### 2b. Sanity check — processed_text samples

In [ ]:
print("=== English sample ===")
for _, row in en_df.head(3).iterrows():
    print(f"  raw : {row['review_text'][:80]}")
    print(f"  proc: {row['processed_text'][:80]}")
    print()

print("=== Vietnamese sample ===")
for _, row in vi_df.head(3).iterrows():
    print(f"  raw : {row['review_text'][:80]}")
    print(f"  proc: {row['processed_text'][:80]}")
    print()